In [0]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder\
    .appName("repartition")\
    .getOrCreate()    

# Create DataFrame from a list (no SparkContext/RDD needed, works on Spark Connect)
data = [1, 2, 3, 4, 5]
df = spark.createDataFrame([(x,) for x in data], ["value"])

# Show elements
print(df.collect())

df2 = df.repartition(8)

print(df.select(F.spark_partition_id()).distinct().count())
print(df2.select(F.spark_partition_id()).distinct().count())


df2 = df.repartition(8, "department")

[Row(value=1), Row(value=2), Row(value=3), Row(value=4), Row(value=5)]
1
4


# Working with RDD



In [0]:
numbers = [10, 20, 30, 40, 50]

# Classic RDD version (only works on Assigned/Classic compute, NOT on Spark Connect):
#   rdd = spark.sparkContext.parallelize(numbers)
#   rdd = rdd.map(lambda x: x * 2)
#   print(rdd.collect())
#
# On Standard/Serverless (Spark Connect) this raises:
#   NotImplementedError: sc is not supported on serverless compute,
#   consider using 'spark' instead.

# Spark Connect (Standard/Serverless) equivalent using the DataFrame API:
df = spark.createDataFrame([(x,) for x in numbers], ["value"]) \
    .withColumn("doubled", F.col("value") * 2)  # rdd.map(x -> x * 2) equivalent

print([row["doubled"] for row in df.collect()])
display(df)

value
10
20
30
40
50


In [0]:
numbers = [10, 20, 30, 40, 50]

from pyspark.sql import functions as F

# RDD and SparkContext are unavailable on Standard/Serverless (Spark Connect) compute:
#   "NotImplementedError: sc is not supported on serverless compute, consider using 'spark' instead."
# So we use the equivalent DataFrame API instead, which works on ALL compute types.

# rdd = spark.sparkContext.parallelize(numbers)  -> create DataFrame from local list
df = spark.createDataFrame([(x,) for x in numbers], ["value"])

# rdd.map(lambda x: x * 2)  -> withColumn expression
df_doubled = df.withColumn("doubled", F.col("value") * 2)

# rdd.reduce(lambda a, b: a + b)  -> aggregate sum
print("Results:", [row["doubled"] for row in df_doubled.collect()])
print("Sum:", df_doubled.select(F.sum("doubled")).collect()[0][0])

df_doubled.display()


Results: [20, 40, 60, 80, 100]
Sum: 300


value,doubled
10,20
20,40
30,60
40,80
50,100


In [0]:
from pyspark.sql import functions as F

data = [
    (1, "Ravi", "IT", 50000, "CLOSED", "Hyderabad"),
    (2, "Arjun", "HR", 35000, "OPEN", "Chennai"),
    (3, "Sita", "IT", 65000, "CLOSED", "Hyderabad"),
    (4, "Priya", "Finance", 45000, "OPEN", "Bangalore"),
    (5, "Kiran", "IT", 70000, "CLOSED", "Chennai"),
    (6, "Anu", "HR", 30000, "CLOSED", "Hyderabad"),
    (7, "Manoj", "Finance", 55000, "OPEN", "Bangalore"),
    (8, "Divya", "IT", 80000, "OPEN", "Hyderabad"),
    (9, "Rahul", "HR", 40000, "CLOSED", "Chennai"),
    (10, "Sneha", "Finance", 60000, "OPEN", "Hyderabad")
]

columns = [
    "id", "name", "department",
    "salary", "status", "city"
]

df = spark.createDataFrame(data, columns)

df.display()

id,name,department,salary,status,city
1,Ravi,IT,50000,CLOSED,Hyderabad
2,Arjun,HR,35000,OPEN,Chennai
3,Sita,IT,65000,CLOSED,Hyderabad
4,Priya,Finance,45000,OPEN,Bangalore
5,Kiran,IT,70000,CLOSED,Chennai
6,Anu,HR,30000,CLOSED,Hyderabad
7,Manoj,Finance,55000,OPEN,Bangalore
8,Divya,IT,80000,OPEN,Hyderabad
9,Rahul,HR,40000,CLOSED,Chennai
10,Sneha,Finance,60000,OPEN,Hyderabad


In [0]:
from pyspark.sql import functions as F

# Transformation 1
df1 = df.filter(
    F.col("salary") > 50000
)

# Transformation 2
df2 = df1.select(
    "name", "department", "salary"
)

# # Transformation 3
df3 = df2.groupBy("department").agg(
    F.avg("salary").alias("avg_salary")
)

# Action
df3.show()

+----------+-----------------+
|department|       avg_salary|
+----------+-----------------+
|        IT|71666.66666666667|
|   Finance|          57500.0|
+----------+-----------------+



In [0]:
from pyspark.sql import functions as F

# Demo data (same employee dataset as earlier)
data = [
    (1, "Ravi", "IT", 50000, "CLOSED", "Hyderabad"),
    (2, "Arjun", "HR", 35000, "OPEN", "Chennai"),
    (3, "Sita", "IT", 65000, "CLOSED", "Hyderabad"),
    (4, "Priya", "Finance", 45000, "OPEN", "Bangalore"),
    (5, "Kiran", "IT", 70000, "CLOSED", "Chennai"),
    (6, "Anu", "HR", 30000, "CLOSED", "Hyderabad"),
    (7, "Manoj", "Finance", 55000, "OPEN", "Bangalore"),
    (8, "Divya", "IT", 80000, "OPEN", "Hyderabad"),
    (9, "Rahul", "HR", 40000, "CLOSED", "Chennai"),
    (10, "Sneha", "Finance", 60000, "OPEN", "Hyderabad")
]
df = spark.createDataFrame(data, ["id", "name", "department", "salary", "status", "city"])

# =====================================================================
# TRANSFORMATIONS (lazy: they only build a plan, nothing runs yet)
# =====================================================================

# 1. select -- pick columns
selected = df.select("name", "department", "salary")

# 2. filter / where -- keep matching rows
it_employees = df.filter(F.col("department") == "IT")
high_paid = df.where(F.col("salary") >= 60000)

# 3. withColumn -- add or replace a column
df_with_raise = df.withColumn("salary_after_raise", F.col("salary") * 1.10)

# 4. withColumnRenamed -- rename a column
df_renamed = df.withColumnRenamed("city", "location")

# 5. drop -- remove columns
df_dropped = df.drop("status", "city")

# 6. distinct / dropDuplicates -- unique rows
unique_cities = df.select("city").distinct()

# 7. orderBy / sort -- sorting
sorted_by_salary = df.orderBy(F.desc("salary"))

# 8. limit -- take first N rows
top3 = df.orderBy(F.desc("salary")).limit(3)

# 9. groupBy + agg -- aggregation
dept_stats = df.groupBy("department").agg(
    F.count("*").alias("emp_count"),
    F.avg("salary").alias("avg_salary"),
    F.max("salary").alias("max_salary")
)

# 10. join -- combine two DataFrames
address = spark.createDataFrame([
    ("Ravi", "Hyderabad"), ("Sita", "Hyderabad"), ("Divya", "Hyderabad")
], ["name", "work_city"])
joined = df.join(address, on="name", how="inner")

# 11. union -- stack DataFrames
combined = it_employees.union(high_paid)

# 12. conditional logic with when/otherwise
df_categorized = df.withColumn(
    "salary_band",
    F.when(F.col("salary") >= 65000, "High")
     .when(F.col("salary") >= 45000, "Medium")
     .otherwise("Low")
)

# NOTE: none of the above triggered any computation -- Spark only
# recorded the lineage (DAG). Nothing runs until an ACTION is called.

print("Transformations defined. Nothing computed yet!")

# =====================================================================
# ACTIONS (eager: they trigger execution of the plan)
# =====================================================================

# 1. show() -- print rows to console
selected.show()

# 2. collect() -- bring all rows to the driver as a list of Rows
rows = it_employees.collect()
print("IT employees:", [row["name"] for row in rows])

# 3. count() -- number of rows
print("Total employees:", df.count())
print("Unique cities:", unique_cities.count())

# 4. take(n) / head(n) -- first n rows to the driver
print("First 2 rows:", df.take(2))

# 5. first() -- single Row
print("First employee:", df.orderBy("id").first()["name"])

# 6. describe() -- summary statistics (returns a DataFrame, .show() is the action)
df.select("salary").describe().show()

# 7. display() -- rich Databricks table output (an action in notebooks)
display(top3)

# 8. write / saveAsTable -- writing results out is also an action
df_categorized.groupBy("salary_band").count().show()

print("\nDepartment stats (transformation + action together):")
dept_stats.show()

print("\nJoined result:")
joined.select("name", "department", "salary", "work_city").show()

print("\nSalary bands:")
df_categorized.select("name", "salary", "salary_band").show(truncate=False)


Transformations defined. Nothing computed yet!
+-----+----------+------+
| name|department|salary|
+-----+----------+------+
| Ravi|        IT| 50000|
|Arjun|        HR| 35000|
| Sita|        IT| 65000|
|Priya|   Finance| 45000|
|Kiran|        IT| 70000|
|  Anu|        HR| 30000|
|Manoj|   Finance| 55000|
|Divya|        IT| 80000|
|Rahul|        HR| 40000|
|Sneha|   Finance| 60000|
+-----+----------+------+

IT employees: ['Ravi', 'Sita', 'Kiran', 'Divya']
Total employees: 10
Unique cities: 3
First 2 rows: [Row(id=1, name='Ravi', department='IT', salary=50000, status='CLOSED', city='Hyderabad'), Row(id=2, name='Arjun', department='HR', salary=35000, status='OPEN', city='Chennai')]
First employee: Ravi
+-------+-----------------+
|summary|           salary|
+-------+-----------------+
|  count|               10|
|   mean|          53000.0|
| stddev|16020.81978759722|
|    min|            30000|
|    max|            80000|
+-------+-----------------+



id,name,department,salary,status,city
8,Divya,IT,80000,OPEN,Hyderabad
5,Kiran,IT,70000,CLOSED,Chennai
3,Sita,IT,65000,CLOSED,Hyderabad


+-----------+-----+
|salary_band|count|
+-----------+-----+
|     Medium|    4|
|        Low|    3|
|       High|    3|
+-----------+-----+


Department stats (transformation + action together):
+----------+---------+------------------+----------+
|department|emp_count|        avg_salary|max_salary|
+----------+---------+------------------+----------+
|        IT|        4|           66250.0|     80000|
|        HR|        3|           35000.0|     40000|
|   Finance|        3|53333.333333333336|     60000|
+----------+---------+------------------+----------+


Joined result:
+-----+----------+------+---------+
| name|department|salary|work_city|
+-----+----------+------+---------+
| Ravi|        IT| 50000|Hyderabad|
| Sita|        IT| 65000|Hyderabad|
|Divya|        IT| 80000|Hyderabad|
+-----+----------+------+---------+


Salary bands:
+-----+------+-----------+
|name |salary|salary_band|
+-----+------+-----------+
|Ravi |50000 |Medium     |
|Arjun|35000 |Low        |
|Sita |65000 |

In [0]:
## repartition() vs coalesce() - comparing partition counts

from pyspark.sql import functions as F

# Same employee dataset (rebuilt so this cell runs standalone)
data = [
    (1, "Ravi", "IT", 50000, "CLOSED", "Hyderabad"),
    (2, "Arjun", "HR", 35000, "OPEN", "Chennai"),
    (3, "Sita", "IT", 65000, "CLOSED", "Hyderabad"),
    (4, "Priya", "Finance", 45000, "OPEN", "Bangalore"),
    (5, "Kiran", "IT", 70000, "CLOSED", "Chennai"),
    (6, "Anu", "HR", 30000, "CLOSED", "Hyderabad"),
    (7, "Manoj", "Finance", 55000, "OPEN", "Bangalore"),
    (8, "Divya", "IT", 80000, "OPEN", "Hyderabad"),
    (9, "Rahul", "HR", 40000, "CLOSED", "Chennai"),
    (10, "Sneha", "Finance", 60000, "OPEN", "Hyderabad")
]
df = spark.createDataFrame(data, ["id", "name", "department", "salary", "status", "city"])

# RDD getNumPartitions() is unavailable on Spark Connect,
# so count partitions via spark_partition_id()
def num_partitions(d):
    return d.select(F.spark_partition_id().alias("pid")).distinct().count()

print("original df partitions    :", num_partitions(df))
print("repartition(8) partitions :", num_partitions(df.repartition(8)))
print("coalesce(1) partitions    :", num_partitions(df.coalesce(1)))

# repartition(n): full shuffle -> can INCREASE or DECREASE the partition count
# coalesce(n):    no shuffle, merges existing partitions -> can only DECREASE
#                 (asking for MORE partitions than exist is a silent no-op)

original df partitions    : 1
repartition(8) partitions : 6
coalesce(1) partitions    : 1


In [0]:
## Where does the shuffle happen? Compare the physical plans

# repartition() inserts an Exchange (REPARTITION_BY_NUM) node -> a FULL SHUFFLE
print("=== repartition(8) plan ===")
df.repartition(8).explain()

print("\n=== coalesce(1) plan ===")
df.coalesce(1).explain()

# In the repartition plan you see an Exchange / hashpartitioning stage:
#   every row is moved across the network to build the new partitions.
# In the coalesce plan there is NO Exchange: partitions are merged
#   within the same stage (a "narrow" dependency).

=== repartition(8) plan ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonShuffleExchangeSource false
         +- PhotonShuffleMapStage REPARTITION_BY_NUM, [id=#7421]
            +- PhotonShuffleExchangeSink RoundRobinPartitioning(8)
               +- PhotonSort [id#10209L ASC NULLS FIRST, name#10210 ASC NULLS FIRST, department#10211 ASC NULLS FIRST, salary#10212L ASC NULLS FIRST, status#10213 ASC NULLS FIRST, city#10214 ASC NULLS FIRST]
                  +- PhotonRowToColumnar
                     +- LocalTableScan [id#10209L, name#10210, department#10211, salary#10212L, status#10213, city#10214]


== Photon Explanation ==
The query is fully supported by Photon.

=== coalesce(1) plan ===
== Physical Plan ==
Coalesce 1
+- LocalTableScan [id#10218L, name#10219, department#10220, salary#10221L, status#10222, city#10223]


== Photon Explanation ==
Photon does not fully support the query because:


In [0]:
## repartition() by column - hash partitioning

# All rows with the same department hash to the SAME partition.
# Useful before joins or aggregations on that column.
by_dept = df.repartition(3, "department")

by_dept.withColumn("partition_id", F.spark_partition_id()) \
    .orderBy("partition_id", "department") \
    .show()

# coalesce() cannot do this - it has no column form,
# because it only merges existing partitions without looking at values.

+---+-----+----------+------+------+---------+------------+
| id| name|department|salary|status|     city|partition_id|
+---+-----+----------+------+------+---------+------------+
|  1| Ravi|        IT| 50000|CLOSED|Hyderabad|           0|
|  3| Sita|        IT| 65000|CLOSED|Hyderabad|           0|
|  5|Kiran|        IT| 70000|CLOSED|  Chennai|           0|
|  8|Divya|        IT| 80000|  OPEN|Hyderabad|           0|
|  2|Arjun|        HR| 35000|  OPEN|  Chennai|           1|
|  6|  Anu|        HR| 30000|CLOSED|Hyderabad|           1|
|  9|Rahul|        HR| 40000|CLOSED|  Chennai|           1|
|  4|Priya|   Finance| 45000|  OPEN|Bangalore|           2|
|  7|Manoj|   Finance| 55000|  OPEN|Bangalore|           2|
| 10|Sneha|   Finance| 60000|  OPEN|Hyderabad|           2|
+---+-----+----------+------+------+---------+------------+



In [0]:
## Typical use case: parallelize wide, coalesce before writing

# 1) repartition(4): spread rows evenly across 4 partitions (full shuffle, balanced)
even = df.repartition(4)
print("partitions after repartition(4):", even.select(F.spark_partition_id()).distinct().count())

# 2) coalesce(1): merge into 1 partition WITHOUT a new shuffle.
#    Classic trick before writing a small result as a single output file.
single = even.coalesce(1)
print("partitions after coalesce(1)   :", single.select(F.spark_partition_id()).distinct().count())

# Rule of thumb:
#   repartition() to RAISE parallelism or balance skewed data
#   coalesce()    to LOWER partition count cheaply (e.g. fewer output files)

partitions after repartition(4): 4
partitions after coalesce(1)   : 1


# repartition() vs coalesce() — cheat sheet

| | `repartition(n)` | `coalesce(n)` |
|---|---|---|
| Shuffle | Full shuffle (new Exchange stage) | None — merges existing partitions |
| Partition count | Can increase or decrease | Can only decrease (larger `n` is a no-op) |
| Data balance | Even, regardless of input skew | Can produce uneven partitions |
| Cost | Expensive — moves all rows over the network | Cheap — narrow dependency |
| By column | Yes — `repartition(n, "col")` hash-partitions | Not supported |
| Typical use | Before join/groupBy, or to raise parallelism | After filtering, or to write fewer files |

**Rules of thumb**

* Increasing parallelism → `repartition()`
* Shrinking to fewer output files before a write → `coalesce()`
* Need evenly sized partitions and can afford the shuffle → `repartition()`